# T77 — Grille secteur × taille × domaine par un LLM, sur Google Colab (GPU)

Note chaque code NAF (718) : une entreprise de ce secteur recrute-t-elle habituellement en data, informatique, finance, marketing, RH, selon sa taille (0 à 3) ? Méthode : `docs/PISTE_DOMAINES_LLM.md`. Le LLM ne voit que le code NAF et son libellé (nomenclature publique) : aucune donnée d'entreprise.

**Avant de commencer**
1. *Exécution → Modifier le type d'exécution → GPU* (T4 en gratuit ; L4 ou A100 avec Colab Pro).
2. Sur ton Google Drive, crée le dossier `radar_llm` et dépose-y `naf_liste.csv` (fichier envoyé par Mathis ; produit par `scripts/build_grille_llm.py --export-naf`).
3. Exécute les cellules **dans l'ordre**, une par une.

Si la session se coupe : réexécute tout depuis le début. Les secteurs déjà notés sont en cache sur Drive et ne sont pas recalculés.

## 1. Carte graphique et Google Drive

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')
DIR = '/content/drive/MyDrive/radar_llm'
import os
assert os.path.exists(f'{DIR}/naf_liste.csv'), "Dépose naf_liste.csv dans MyDrive/radar_llm"

## 2. Code du projet (dépôt public) et Ollama

In [ ]:
!rm -rf /content/hackathon && git clone -q --depth 1 https://github.com/MathisKacer/hackathon /content/hackathon
!pip -q install python-dotenv pyarrow
!apt-get -qq install -y zstd pciutils > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null && ollama --version

## 3. Choix du modèle selon la mémoire de la carte, démarrage du serveur

| Mémoire | Modèle | Requêtes en parallèle |
|---|---|---|
| < 20 Go (T4) | `qwen2.5:14b` (compressé, ~9 Go) | 4 |
| 20–35 Go (L4) | `qwen2.5:32b` (compressé, ~20 Go) | 2 |
| ≥ 35 Go (A100) | `qwen2.5:32b` | 8 |

In [ ]:
import subprocess, time
mem = int(subprocess.check_output(
    ['nvidia-smi', '--query-gpu=memory.total', '--format=csv,noheader,nounits']).split()[0])
if mem < 20000:
    MODEL, WORKERS = 'qwen2.5:14b', 4
elif mem < 35000:
    MODEL, WORKERS = 'qwen2.5:32b', 2
else:
    MODEL, WORKERS = 'qwen2.5:32b', 8
print(f'{mem} Mo → {MODEL}, {WORKERS} requêtes en parallèle')

env = dict(os.environ, OLLAMA_NUM_PARALLEL=str(WORKERS))
server = subprocess.Popen(['ollama', 'serve'], env=env,
                          stdout=open('/content/ollama.log', 'w'), stderr=subprocess.STDOUT)
time.sleep(5)
!ollama pull {MODEL}

## 4. Test sur les 20 secteurs les plus fréquents (~2 à 5 min)

Ce sont les mêmes 20 secteurs que le test précédent, pour comparer. La consigne utilisée est la dernière version du dépôt (v3) ; `--consigne v1` pour l'ancienne. Chaque version a son propre cache.

In [ ]:
%cd /content/hackathon
!PYTHONPATH=. python scripts/build_grille_llm.py --naf-csv {DIR}/naf_liste.csv --cache-dir {DIR}/cache \
    --model {MODEL} --workers {WORKERS} --limit 20 --out {DIR}/test20.parquet

**Vérifier les notes** (petits et grands établissements). Repères attendus en 1 000 et plus : banques (64.19Z) finance 3 ; supermarchés finance et RH 2 ; administration RH ≥ 2 ; restauration data 0 ; banques finance 3 dès 50–249 (cœur de métier) ; **peu de 3** hors cœur de métier (maçonnerie, nettoyage : marketing ≤ 1). En 10–49 : surtout des 0 et des 1. Si c'est grossièrement faux, prévenir Mathis avant de lancer la suite.

In [ ]:
import pandas as pd
t = pd.read_parquet(f'{DIR}/test20.parquet')
print(t['modele'].iloc[0], 'consigne', t['consigne'].iloc[0])
for taille in ['10–49', '1 000 et plus']:
    print(f'
— établissements de {taille} salariés —')
    print(t[t.taille == taille].pivot_table(index=['naf', 'naf_libelle'], columns='domaine',
          values='note_llm')[['data', 'informatique', 'finance', 'marketing', 'rh']].to_string())

In [ ]:
# Justifications d'un secteur (changer le code)
t[(t.naf == '64.19Z') & (t.taille == '1 000 et plus')][['domaine', 'note_llm', 'justification']]

## 5. Calcul complet (T4 + 14B : ~18 s par secteur)

`LIMIT = 360` : les 360 secteurs les plus fréquents (95 % des établissements), ~1 h 50 sur une T4. `LIMIT = None` : les 718 (~3 h 30). Les secteurs manquants recevront la moyenne de leur division NAF.

Rester sur l'onglet (Colab coupe les sessions inactives). En cas de coupure : réexécuter les cellules 1 à 3, puis celle-ci (le cache reprend où on en était).

In [ ]:
LIMIT = 360
lim = f'--limit {LIMIT}' if LIMIT else ''
!PYTHONPATH=. python scripts/build_grille_llm.py --naf-csv {DIR}/naf_liste.csv --cache-dir {DIR}/cache \
    --model {MODEL} --workers {WORKERS} {lim} --out {DIR}/grille_domaines.parquet

## 6. Résultat

`MyDrive/radar_llm/grille_domaines.parquet` (+ le dossier `cache/`, une réponse JSON par secteur). Envoyer `grille_domaines.parquet` à Mathis, ou le télécharger avec la cellule ci-dessous.

In [ ]:
g = pd.read_parquet(f'{DIR}/grille_domaines.parquet')
print(g['naf'].nunique(), 'secteurs notés ;', g['modele'].iloc[0])
print(g.groupby('domaine')['note_llm'].value_counts().unstack(fill_value=0))
from google.colab import files
files.download(f'{DIR}/grille_domaines.parquet')